# Actividad 2.3 — Validación estructural y semántica de datos

## Objetivo
En este notebook trabajaremos la tercera etapa del pipeline de datos: la validación estructural y semántica.

A diferencia de las etapas anteriores, aquí no volveremos a realizar la ingesta ni la transformación, ya que ese proceso ya fue ejecutado previamente y el dataset resultante se encuentra en la carpeta `data/processed/`.

## Flujo del pipeline trabajado
1. Ingesta de datos
2. Limpieza y transformación
3. Validación estructural y semántica ← etapa actual

## Meta de esta actividad
- Leer el dataset procesado
- Aplicar validaciones estructurales
- Aplicar validaciones semánticas
- Separar registros válidos e inválidos
- Registrar observaciones en logs
- Exportar resultados para la siguiente etapa del pipeline

## Contexto de trabajo

En este proyecto ya existe una etapa previa de limpieza y transformación. Por lo tanto, este notebook asume que el dataset de entrada ya fue generado y almacenado en la carpeta:

`../data/processed/`

A partir de ese archivo, se verificará si los datos cumplen con reglas técnicas y lógicas antes de ser cargados posteriormente a una base de datos.

## Carga del dataset procesado

En esta etapa leeremos el archivo generado en la fase anterior del pipeline.

Si tu archivo tiene otro nombre, debes ajustar la ruta en la siguiente celda.

In [ ]:
import pandas as pd
import os
import logging
import numpy as np

ruta_entrada = "../data/processed/ventas_clean.csv"

df = pd.read_csv(ruta_entrada)
df


In [ ]:
df2 = pd.read_csv("./ventas_raw.csv")
df2

## Exploración inicial

Antes de validar, revisaremos:
- estructura general del dataset
- tipos de datos
- posibles nulos remanentes

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## Configuración del log

Crearemos un archivo de log para dejar trazabilidad del proceso de validación.

In [ ]:
os.makedirs("../logs", exist_ok=True)

logging.basicConfig(
    filename="../logs/validation.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Inicio del proceso de validación del dataset procesado")
print("Logger configurado correctamente")

## Reglas de validación

Aplicaremos dos tipos de validación:

### Validaciones estructurales
1. La fecha debe tener formato válido
2. La cantidad debe ser numérica
3. El precio debe ser numérico

### Validaciones semánticas
1. La cantidad debe ser mayor a 0 y razonable
2. El precio debe ser mayor a 0
3. Si existe descuento, debe estar entre 0 y 100
4. Producto y ciudad no deben estar vacíos

In [ ]:
def validar_fecha(valor):
    try:
        pd.to_datetime(valor, format="%Y-%m-%d")
        return True
    except:
        return False


def validar_numerico(valor):
    try:
        float(valor)
        return True
    except:
        return False


def validar_cantidad_semantica(valor):
    try:
        valor = float(valor)
        return 0 < valor <= 500
    except:
        return False


def validar_precio_semantico(valor):
    try:
        valor = float(valor)
        return valor > 0
    except:
        return False


def validar_descuento(valor):
    if pd.isnull(valor):
        return True
    try:
        valor = float(valor)
        return 0 <= valor <= 100
    except:
        return False


def validar_texto_no_vacio(valor):
    return pd.notnull(valor) and str(valor).strip() != ""

In [ ]:
df["descuento"] = np.nan

df

In [ ]:
df3 = df.copy()
df3

## Aplicación de validaciones

Generaremos columnas booleanas para identificar si cada registro cumple o no con cada regla.

In [ ]:
df_validacion = df.copy()

# Validaciones estructurales
df_validacion["val_fecha"] = df_validacion["fecha"].apply(validar_fecha)
df_validacion["val_cantidad_tipo"] = df_validacion["cantidad"].apply(validar_numerico)
df_validacion["val_precio_tipo"] = df_validacion["precio"].apply(validar_numerico)

# Validaciones semánticas
df_validacion["val_cantidad_negocio"] = df_validacion["cantidad"].apply(validar_cantidad_semantica)
df_validacion["val_precio_negocio"] = df_validacion["precio"].apply(validar_precio_semantico)

if "descuento" in df_validacion.columns:
    df_validacion["val_descuento"] = df_validacion["descuento"].apply(validar_descuento)
else:
    df_validacion["val_descuento"] = True

df_validacion["val_producto"] = df_validacion["producto"].apply(validar_texto_no_vacio)
df_validacion["val_ciudad"] = df_validacion["ciudad"].apply(validar_texto_no_vacio)

df_validacion.head()

## Consolidación de la validación

Un registro será considerado válido solo si cumple todas las reglas definidas.

In [ ]:
columnas_validacion = [
    "val_fecha",
    "val_cantidad_tipo",
    "val_precio_tipo",
    "val_cantidad_negocio",
    "val_precio_negocio",
    "val_descuento",
    "val_producto",
    "val_ciudad"
]

df_validacion["registro_valido"] = df_validacion[columnas_validacion].all(axis=1)

df_validacion[["registro_valido"] + columnas_validacion].head()

## Generación de observaciones

Para cada registro inválido, construiremos un detalle del motivo del rechazo.

In [ ]:
def obtener_observaciones(row):
    errores = []

    if not row["val_fecha"]:
        errores.append("Fecha inválida")
    if not row["val_cantidad_tipo"]:
        errores.append("Cantidad no numérica")
    if not row["val_precio_tipo"]:
        errores.append("Precio no numérico")
    if row["val_cantidad_tipo"] and not row["val_cantidad_negocio"]:
        errores.append("Cantidad fuera de rango lógico")
    if row["val_precio_tipo"] and not row["val_precio_negocio"]:
        errores.append("Precio inválido o menor/igual a 0")
    if not row["val_descuento"]:
        errores.append("Descuento fuera de rango 0-100")
    if not row["val_producto"]:
        errores.append("Producto vacío o nulo")
    if not row["val_ciudad"]:
        errores.append("Ciudad vacía o nula")

    return " | ".join(errores) if errores else "Registro válido"

df_validacion["observaciones"] = df_validacion.apply(obtener_observaciones, axis=1)

df_validacion[["observaciones", "registro_valido"]].head()

## Separación de registros válidos e inválidos

In [ ]:
df_validos = df_validacion[df_validacion["registro_valido"]].copy()
df_invalidos = df_validacion[~df_validacion["registro_valido"]].copy()

print("Cantidad de registros válidos:", len(df_validos))
print("Cantidad de registros inválidos:", len(df_invalidos))

## Registros válidos

In [ ]:
df_validos

## Registros inválidos

In [ ]:
columnas_salida = [col for col in df.columns if col in df_invalidos.columns]
columnas_salida += ["registro_valido", "observaciones"]

df_invalidos[columnas_salida]

## Registro del proceso en logs

In [ ]:
logging.info(f"Total registros procesados: {len(df_validacion)}")
logging.info(f"Total registros válidos: {len(df_validos)}")
logging.info(f"Total registros inválidos: {len(df_invalidos)}")

for _, row in df_invalidos.iterrows():
    logging.warning(f"Registro inválido detectado: {row['observaciones']}")

print("Información registrada en ../logs/validation.log")

## Exportación de resultados

Los archivos generados en esta etapa quedarán en la carpeta `processed`, ya que son resultados del pipeline listos para la siguiente fase.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

df_validos.to_csv("../data/processed/ventas_validas.csv", index=False)
df_invalidos.to_csv("../data/processed/ventas_invalidas.csv", index=False)

print("Archivos exportados correctamente en ../data/processed/")

## Reflexión final

### Preguntas de análisis
1. ¿Qué diferencia existe entre una validación estructural y una semántica?
2. ¿Qué errores fueron detectados en esta etapa que no necesariamente se corrigen en transformación?
3. ¿Por qué es importante separar registros válidos de inválidos?
4. ¿Qué riesgos existirían si cargamos datos inválidos directamente a una base de datos?

## Conclusión
Este notebook representa la etapa de validación dentro del pipeline de datos y toma como entrada el resultado generado previamente en `processed`. De este modo, el flujo completo queda organizado por etapas y cada notebook cumple una función específica dentro del proyecto.